<table style="background-color:#F5F5F5;" width="100%">
<tr><td style="background-color:#F5F5F5;"><img src="../images/logo.png" width="150" align='right'/></td></tr>     <tr><td>
            <h2><center>Aprendizagem Automática em Engenharia Biomédica</center></h2>
            <h3><center>1st Semester - 2026/2027</center></h3>
            <h4><center>Universidade Nova de Lisboa - Faculdade de Ciências e Tecnologia</center></h4>
</td></tr>
    <tr><td><h2><b><center>Lab 3 - Cross Validation </center></b></h2>
    <h4><i><b><center>Human Activity Recognition with Decision Trees</center></b></i></h4></td></tr>
</table>

## 1. Introduction to Optimizing Machine Learning Models

In machine learning, building a predictive model is only the first step toward solving a real-world problem. To ensure the model performs well in practice, it is essential to optimize it by finding the best combination of model parameters and structure. This process of __model optimization__ helps us enhance a model's ability to generalize, meaning it performs well on new, unseen data.

### 1.1 The Importance of Model Optimization

Model optimization allows us to achieve three key goals:

* __Generalization__: A well-optimized model should generalize effectively across various datasets. Generalization refers to the model’s ability to perform well not just on the training data but also on new data it hasn't seen before.
* __Model Selection__: Optimization allows us to systematically compare different models and select the one that performs best for the task at hand.
* __Avoiding Overfitting and Underfitting__:
    * __Overfitting__: occurs when a model learns the noise in the training data instead of the underlying patterns. This results in high performance on training data but poor performance on new data.
    * __Underfitting__: happens when the model is too simple to capture the patterns in the data, resulting in low performance on both training and testing data.

### 1.2 Scikit-Learn Package

The [Scikit-learn](https://scikit-learn.org/stable/index.html) library is the standard tool to train end-to-end ML pipelines. It offers a set of tools that range from data preparation to model training and evaluation.

Its methods are highly consistent between algorithms and provide an easy user interface:

* __Estimators__: Includes all objects that can estimate some set of paramenter based on a training set. The process od estimation is performed by the `fit()` method. When calling for an estimator, one can define a set of additional parameters to guide the training/fitting process, which are called hyperparameters.

* __Transformers__: Some estimators can also transform a dataset, especially with data preparation tools. These can be called by the `transform()` method, which resorts to previously fitted parameters to transform the data. The `fit_transform()` method is also available, to perform both actions in one method.

* __Predictors__: On the other hand, some estimators can make predictions on new data, after being fitted on the training set. The new data is sent to the `predict()` method, which returns the predictions. One can also call the `score()` method, which measures the quality of the predictions on the test set.

Also, `sklearn` is built upon existing libraries, such as `Numpy`, which allows to use common array procedures. 


### 1.3 Human Activity Recognition
Human Activity Recognition (HAR) is a growing field in machine learning where the goal is to classify physical activities based on data from inertal sensors (such as accelerometers, gyroscopes, magnetometers), video (RGB, depth or thermal cameras), audio (microphones), among others. Data collected from these sensors can be used to infer activities like walking, running, sitting, and more.

HAR has a wide range of applications, such as:
* __Healthcare__: monitoring patient activity and rehabilitation progress
* __Fitness__: tracking exercise routines
* __Smart Homes__: detecting daily activities for automation

Since such activities change continuously, it is necessary to consider data collected throughout time, so one can predict the performed activity at any given moment.

#### 1.3.1. The UCI Human Activity Recognition Using Smartphones Dataset

In today's class we are going to utlize a widely-used dataset for HAR: the [__UCI Human Activity Recognition Using Smartphones Dataset__](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones):

The UCI Human Activity Recognition Using Smartphones Dataset was collected from __30 participants__ (ageed between 19 and 48 years old) performing __six different activities__ while wearing a __smartphone on their waist__. The smartphone recorded data from its embedded accelerometer and gyroscope sensors at a constant rate of __50Hz__, capturing __three-dimensional linear acceleration and angular velocity__.

The performed activites and their corresponding label are:

| Activity             | Label  |
|----------------------|--------|
| Walking              | 1      |
| Walking upstairs     | 2      |
| Walking downstairs   | 3      |
| Sitting              | 4      |
| Standing             | 5      |
| Laying               | 6      |


The dataset is already divided into __train__ and __test__ sets. These are setup as follows:
1. __Training Set__: 70% of the participants used for model training (a total of 21 pariticpants).
2. __Test Set__: 30% of the participants used for model evaluation (a total of 9 participants).

#### 1.3.2 Downloading the Dataset

The code below will download the dataset into your project.


In [1]:
import requests
import zipfile
from io import BytesIO

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip"

response = requests.get(url)

with zipfile.ZipFile(BytesIO(response.content)) as z:
    z.extractall("Data/")

In [2]:
import numpy as np
import pandas as pd

X_train = pd.read_csv("Data/UCI HAR Dataset/train/X_train.txt", sep=r"\s+", header=None)
X_test = pd.read_csv("Data/UCI HAR Dataset/test/X_test.txt", sep=r"\s+", header=None)

y_train = np.array(pd.read_csv("Data/UCI HAR Dataset/train/y_train.txt", header=None)).ravel()
y_test = np.array(pd.read_csv("Data/UCI HAR Dataset/test/y_test.txt", header=None)).ravel()

activity_labels = pd.read_csv("Data/UCI HAR Dataset/activity_labels.txt", sep=r"\s+", header=None, names=["id", "activity"])

subject_train = np.array(pd.read_csv("Data/UCI HAR Dataset/train/subject_train.txt", header=None, names=["subject"])).ravel()
subject_test = np.array(pd.read_csv("Data/UCI HAR Dataset/test/subject_test.txt", header=None, names=["subject"])).ravel()

### 1.4 Using a Classifier in Scikit-learn

Before comparing several classifiers with cross-validation, let us first look at the basic workflow used by scikit-learn classifiers.

Most classifiers follow the same interface:

```python
model = Classifier(...)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
score = model.score(X_test, y_test)
```

__Exercise 1__:

Use a `DecisionTreeClassifier` to classify the activities in the HAR dataset.

__Tasks__:
* Create a `DecisionTreeClassifier` instance.
* Train the model using `X_train` and `y_train` with `.fit()`.
* Use `.predict()` to obtain predictions for `X_test`.
* Calculate the test accuracy using `accuracy_score`.
* Calculate the same accuracy using the model's `.score()` method.
* Check that the two values are the same.

__Hint__: Remember that `accuracy_score` receives the true labels first and the predicted labels second.


## 2. Cross Validation

To ensure our model generalizes well and isn’t overfitted to training data, we need more than just performance metrics.

When we use a single train-test split to compare models, we risk choosing the model with the best test performance by chance, leading to a biased evaluation. This approach can compromise the impartiality of our test set.

One strategy to avoid this problem is to apply __Cross Validation (CV)__ while training. This process essentially consists on training a model multiple times, with different training sets, to evaluate if its performance is robust to slight variations. This means that we are essentially seperating our training dataset into several subsets consisiting of training and validation sets.

<div>
<img src="../images/cross_validation.png" width="700"/>
</div>

### 2.1 Purpose of Cross Validation

CV allows us to:

* __Improve Reliability__: By using multiple splits, cross validation provides a more reliable estimate of a model’s performance.
* __Select Models and Parameters__: CV enables us to compare various models or configurations systematically to choose the most suitable one.
* __Prevent Overfitting and Underfitting__: Since CV evaluates the model across multiple subsets, it reduces the risk of tuning our model to a single train-test split, leading to better generalization.

### 2.2 Types of Cross Validation
There are different strategies one can adopt. Each of these methods offers distinct advantages depending on the dataset structure and the specific goals of the model evaluation. Understanding which cross validation approach to apply helps ensure a more reliable model performance estimate, building a foundation for effective hyperparameter tuning in the following steps. The list below presents some CV techniques. The list does not prentend to be complete.

__1. K-Fold Cross Validation__:

* __Overview__: The dataset is divided into K approximately equally-sized “folds.” The model trains on K-1 folds and is evaluated on the remaining fold. This process is repeated K times, with each fold serving as the validation set once. The final performance is the average of all K validation results.
* __Use Case__: K-Fold CV is widely used in general machine learning problems, offering a balanced trade-off between computational efficiency and reliable performance estimation.
* __sklearn function__: [__KFold()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html#sklearn.model_selection.KFold)

__2. Stratified K-Fold Cross Validation__:

* __Overview__: Similar to K-Fold CV, but attempts to preserve the class proportions in each fold. This is particularly useful for classification problems.
* __Use Case__: Preferred for classification tasks where maintaining class balance in each fold is essential for fair evaluation.
* __sklearn function__: [__StratifiedKFold()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)

__3. Leave-One-Out Cross Validation (LOOCV)__:

* __Overview__: Each data point acts as a single test set, with all other data points used for training. The process is repeated for each data point, making LOOCV computationally expensive but providing a very fine-grained performance estimate.
* __Use Case__: Best suited for small datasets where having a robust estimate of model performance is crucial.
* __sklearn function__: [__LeaveOneOut()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneOut.html#sklearn.model_selection.LeaveOneOut)

__4. Shuffled Cross Validation__:

* __Overview__: This approach involves randomly shuffling the dataset, then splitting it into training and testing subsets. This process is repeated multiple times, with a new shuffle each time. Unlike K-Fold, Shuffled CV doesn’t ensure equal-sized splits or folds but allows flexibility in the size of train-test splits.
* __Use Case__: Shuffled CV is helpful when a specific proportion (e.g., 70% train, 30% test) is preferred, offering a broader view of model performance across multiple random partitions.
* __sklearn function__: [__ShuffleSplit()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ShuffleSplit.html#sklearn.model_selection.ShuffleSplit)

__5. TimeSeriesSplit__:
* __Overview__: Designed specifically for time-series data, this method splits the data based on a sequence, maintaining the temporal order. Each fold trains on an earlier segment of the data and tests on a later segment, reflecting a realistic forecasting scenario.
* __Use Case__: Ideal for time-series forecasting or sequential data where the order of data points matters.
* __sklearn function__: [__TimeSeriesSplit()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)

__6. GroupKFold__:

* __Overview__: In situations where data points are grouped by a common identifier (e.g., data from the same subject or experiment), GroupKFold ensures that all data points from a particular group are kept together within either the training or test set. This avoids data leakage from having related data points in both the training and test sets. GroupKFold is a variation of K-Fold Cross Validation specifically for grouped data. This approach is similar to K-Fold but ensures that groups are not split across folds.
* __Use Case__: Ideal for datasets with identifiable groupings, such as repeated measurements from the same subjects or multiple instances from similar sources. GroupKFold provides a reliable performance estimate while respecting group structures, making it a common choice in healthcare, behavioral, and customer studies where data points within groups may be highly correlated.
* __sklearn function__: [__GroupKFold()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)

### 2.3 Applying Simple Cross Validation Using Scikit-learn

Scikit-learn provides several convenience functions to simplify the implementation of cross-validation in machine learning workflows. These functions allow us to efficiently assess model performance without manually coding the cross-validation logic. By leveraging these built-in utilities, we can focus on model development while ensuring robust evaluation through established cross-validation techniques.

__1. cross_val_score__:
* __Overview__: the cross_val_score function provides a simple and efficient way to evaluate a model's performance through cross-validation. If the estimator is a classifier and `y` is binary or multiclass, using an integer such as `cv=10` makes scikit-learn use `StratifiedKFold` by default. Other CV strategies can be explicitly passed through the `cv` parameter.
* __Use Case__: cross_val_score is particularly useful when you want a quick assessment of a model's performance across different validation folds without manually implementing the CV loop. It returns one score per fold, which can then be summarized using the mean and standard deviation.
* __sklearn function__: [__cross_val_score()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)

__2. cross_validate__:
* __Overview__: the cross_validate function extends the capabilities of cross_val_score by allowing the computation of multiple evaluation metrics in a single call. It also returns detailed information about the training and validation scores, enabling a comprehensive analysis of model performance.
* __Use Case__: cross_validate is beneficial when you want to evaluate a model based on various performance metrics (e.g., accuracy, F1 score, recall) simultaneously. This function is particularly useful in complex modeling scenarios where understanding different aspects of model performance is critical. It is a good choice when you require detailed insights, such as training times and the ability to analyze multiple metrics, to guide your model selection and tuning process.
* __sklearn function__: [__cross_validate()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html)


__Exercise 2__:  
Before we can apply CV techniques, we need to prepare some models to compare them to each other. To the models it is convenient to keep them together in a single data structure, which can be a list of tuples containing the model name and the corresponding model.

__Tasks__:
* Create a list, named __models__, containing a tuple consisitng of the __name of the classifier__ and an  __instance of the classifier__ itself.
* From [sklearn supervised learning algorithms](https://scikit-learn.org/stable/supervised_learning.html#supervised-learning) select at least one classifier from each of the following groups: 
1. Support Vector Machines
2. Nearest Neighbors
3. Naive Bayes
4. Decision Trees
5. Ensemble methods

Example: `models = [("DecisionTree", DecisionTreeClassifier()), ("Clf name", clf() )]`

### 2.4 Understanding What Cross-Validation Is Doing

`cross_val_score()` hides the loop that creates the training and validation folds. Before using it, let us inspect those folds directly.

__Exercise 3__:

Use `StratifiedKFold` with 5 folds to inspect how the training data is divided.

__Tasks__:
* Create a `StratifiedKFold` object with `n_splits=5`.
* Use `.split(X_train, y_train)` to obtain the training and validation indices for each fold.
* For each fold, print the number of samples in the training and validation sets.
* Print the class proportions in the validation set.
* Check that every sample is used for validation exactly once.

__Question__: Why might `StratifiedKFold` be preferable to ordinary `KFold` for a classification problem?


__Exercise 4__: 

Now that we have defined the models that we are going to use, we need to write a function that performs the training and evaluation steps for us. This function can then be used in a for-loop and in each iteration we pass one of the defined models to the function. By writing these types of abtsracted fucntions we can write an efficent ML pipeline.

__Tasks__:
* Implement a function that performs training and evaluation of a model.
* The function should be called: __train_and_evaluation__.
* The function should have the following inputs:
    >* __model__: the tuple containng the model name and the instance of the model | Type: Tuple[str, Any]
    >* __X_train__: training features as a numpy array | Type: np.ndarray
    >* __y_train__: training labels as a numpy array | Type: np.ndarray
    >* __X_test__: test features as numpy array | Type: np.ndarray
    >* __y_test__: test labels as numpy array | Type: np.ndarray
* The function should output:
    >* __metrics__: a dictionary containing the following keys (with their corresponding values)   
    >  '_Model_': the name of the model  
    >  '_Train ACC_': the accuracy on the entire training set  
    >  '_Test ACC_': the accuracy on the test set  
    >  '_K-fold ACC mean_': the mean accuracy over all folds  
    >  '_K-fold ACC std_': the standard deviation of the accuracy over all folds  
* Use the [cross_val_score()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) function with a K-fold of 10

__Hints__:
* For 10-fold cross-validation on this classification problem, setting `cv=10` makes scikit-learn use `StratifiedKFold` by default.
* The cross_val_score() function returns an numpy.ndarray containing the accuracy for each fold.
* All scikit-learn models have a `.score(X, y)` function that calculates the accuracy of the model with respect to the provided X and y. This way you do not have to use the accuracy_score() function. The `.score(X, y)` should be used after the model has been trained.

__Exercise 5__: 

Now, it is time to put your function to work.

__Tasks__:

* Use the previous function to train and evaluate the classifiers from __Exercise 2__.
* Collect all results in a list.
* Transform that list into pandas.DataFrame.
* Set "Model" as the index of the pandas.DataFrame.
* Print the DataFrame.

### 2.5 Implementing a GroupKFold in Scikit-learn

While the K-Fold strategy shown above works well when all training instances are independent, it’s not ideal for the HAR dataset, where this assumption doesn’t hold. In the HAR dataset:

* The dataset contains many observations from the same subject. These observations can share subject-specific patterns.
* If observations from the same subject appear in both training and validation sets, the validation score may partly reflect how well the model recognizes that subject rather than how well it generalizes to a new subject.
  
Because of these properties, __each subject’s data should be treated as a distinct group__. When we want to estimate performance on previously unseen subjects, all observations from a subject should remain together in either the training or validation set. This prevents subject-level leakage and makes the validation scenario closer to the real-world question of generalization to a new person.

Therefore, we will use the [__GroupKFold()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) strategy to split our data into 10 folds. We will only apply this process to the train set with the statistical features, while the original test set will be left for evaluation. Let's see how it works. 

In [ ]:
from sklearn.model_selection import GroupKFold

# intialize the GroupKFold using 10 splits
group_kfold = GroupKFold(n_splits=10)

# check how many splits were established by the CV strategy (not really necessary this is just to show how to retrieve this information)
print("The GroupKFold was set to have {} splits".format(group_kfold.get_n_splits(X_train, y_train, subject_train)))

# see how the subjects were distributed for each fold
for fold, (train_index, val_index) in enumerate(group_kfold.split(X_train, y_train.ravel(), subject_train), start=1):

    # print the subjects used for training and testing
    print("Subjects for training: {} | Subjects for testing: {}".format(np.unique(subject_train[train_index]), np.unique(subject_train[val_index])))

### 2.6 Does the Cross-Validation Strategy Matter?

We have now used two different validation strategies on the same training data. The next step is to understand what question each strategy is answering.

__Exercise 6__:

Use a `LinearSVC` classifier and compare:

1. The default cross-validation obtained with `cv=10`.
2. A `GroupKFold` with 10 folds, using `subject_train` as the grouping variable.

__Tasks__:
* Calculate the accuracy for every fold for both strategies.
* Calculate the mean and standard deviation for each strategy.
* Present the results in a small pandas DataFrame.

__Questions__:
* Why can the two validation strategies produce different results?
* In the default strategy, can the same subject appear in both training and validation?
* In the GroupKFold strategy, can the same subject appear in both training and validation?
* If the goal is to estimate performance on a completely new subject, what does each strategy simulate?


### 2.7 Leave-One-Subject-Out Validation

`GroupKFold` keeps subjects together, but it can still place several subjects in the validation set at the same time. Another possibility is to leave exactly one subject out at each iteration.

This can be implemented with `LeaveOneGroupOut`.

__Exercise 7__:

* Create a `LeaveOneGroupOut` object.
* Use it with `cross_val_score()` and `subject_train` as the groups.
* How many validation iterations are performed?
* What does one validation iteration represent in this dataset?
* Calculate the mean and standard deviation of the accuracy.
